# Momento 4 · Aumento de datos en CIFAR-10

**Objetivo.** Comparar una CNN de referencia sin aumento de datos con la misma CNN entrenada bajo transformaciones individuales y combinadas, manteniendo constantes los datos, la partición, el optimizador y el criterio de parada. La prueba oficial se usa únicamente para evaluación.

**Relación con el Momento 3.** La CNN y el protocolo provienen de `notebooks/reference_momento3.ipynb`. Allí el modelo ya incorporaba reflejo horizontal, rotación y zoom. Por eso la referencia de este experimento es **esa misma CNN sin el bloque estocástico**; `full` reincorpora exactamente esas tres transformaciones. La precisión mixta usada en Colab se desactiva aquí para todas las condiciones por igual.

El cuaderno es autónomo: todas las funciones necesarias se definen abajo. `src/` y `scripts/` conservan el ejecutor por lotes, y las ejecuciones completas se reutilizan únicamente si su manifiesto coincide.

## 1. Configuración y entorno

Se fija la semilla 42, un máximo de 30 épocas y los siete escenarios. `FORCE_RETRAIN=False` reutiliza una ejecución íntegra con manifiesto idéntico; si falta, la entrena. Para verificar sin entrenar se pueden definir `NB_SCENARIOS=reference,full,flip_h,rotation` y `NB_REUSE_ONLY=1` en el entorno. Las rutas se resuelven desde la raíz del repositorio, también al ejecutar desde `notebooks/`.

In [ ]:
from pathlib import Path
from collections import defaultdict
import gc
import hashlib
import json
import os
import pickle
import platform
import random
import sys
import time
import zipfile

ROOT = next((path for path in (Path.cwd(), Path.cwd().parent) if (path / "docs" / "DATASET.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks/ directory")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".cache" / "matplotlib")
os.environ["XDG_CACHE_HOME"] = str(ROOT / ".cache")

import cv2
import imagehash
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import Image, display
from PIL import Image as PILImage
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

SEED = 42
ALL_SCENARIOS = ("reference", "full", "flip_h", "rotation", "zoom", "flip_v", "blur")
SCENARIOS = tuple(filter(None, os.getenv("NB_SCENARIOS", ",".join(ALL_SCENARIOS)).split(",")))
assert SCENARIOS and len(set(SCENARIOS)) == len(SCENARIOS) and set(SCENARIOS) <= set(ALL_SCENARIOS)
EPOCHS = 30
BATCH_SIZE = 128
FORCE_RETRAIN = False
REUSE_ONLY = os.getenv("NB_REUSE_ONLY", "0") == "1"
SUBSET_PER_CLASS = None
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
DATA_DIR = ROOT / "data" / "cifar-10-batches-py"
CLASS_NAMES = np.array(["Avión", "Automóvil", "Pájaro", "Gato", "Ciervo", "Perro", "Rana", "Caballo", "Barco", "Camión"])
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.random.set_seed(SEED)
print(f"Python {platform.python_version()} | TensorFlow {tf.__version__} | devices: {tf.config.list_physical_devices()}")
print(f"Scenarios: {SCENARIOS} | epochs: {EPOCHS} | force retrain: {FORCE_RETRAIN} | reuse only: {REUSE_ONLY}")

## 2. Carga y procedencia

Se lee primero la distribución local Python de CIFAR-10 (`data/cifar-10-batches-py`). Si falta, se recurre a `tf.keras.datasets.cifar10.load_data()`; en Colab, esta opción puede descargar datos la primera vez. La procedencia, composición y limitaciones están documentadas en [`docs/DATASET.md`](../docs/DATASET.md). La prueba oficial permanece separada de las 50 000 imágenes oficiales de entrenamiento.

In [ ]:
def read_batch(path):
    with path.open("rb") as stream:
        batch = pickle.load(stream, encoding="bytes")
    pixels = np.asarray(batch[b"data"], dtype=np.uint8).reshape(-1, 3, 32, 32)
    images = pixels.transpose(0, 2, 3, 1).copy()
    labels = np.asarray(batch[b"labels"], dtype=np.int64)
    if len(images) != len(labels):
        raise ValueError(f"Mismatched images and labels in {path}")
    return images, labels

def load_cifar10(directory):
    required = [*(directory / f"data_batch_{index}" for index in range(1, 6)), directory / "test_batch", directory / "batches.meta"]
    if all(path.is_file() for path in required):
        parts = [read_batch(directory / f"data_batch_{index}") for index in range(1, 6)]
        train_images = np.concatenate([part[0] for part in parts])
        train_labels = np.concatenate([part[1] for part in parts])
        test_images, test_labels = read_batch(directory / "test_batch")
        with (directory / "batches.meta").open("rb") as stream:
            metadata = pickle.load(stream, encoding="bytes")
        if len(metadata[b"label_names"]) != len(CLASS_NAMES):
            raise ValueError("Unexpected CIFAR-10 class metadata")
        provenance = "local Python batches"
    else:
        (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.cifar10.load_data()
        train_labels = train_labels.reshape(-1).astype(np.int64)
        test_labels = test_labels.reshape(-1).astype(np.int64)
        provenance = "tf.keras.datasets.cifar10 (fallback)"
    return train_images, train_labels, test_images, test_labels, provenance

train_raw, train_labels_full, test_raw, test_labels, provenance = load_cifar10(DATA_DIR)
assert train_raw.shape == (50000, 32, 32, 3) and test_raw.shape == (10000, 32, 32, 3)
print(f"Source: {provenance}; official train: {train_raw.shape}; official test: {test_raw.shape}")

## 3. Partición reproducible

Se replica la división estratificada 80/20 del Momento 3 con `random_state=42`: 40 000 imágenes de entrenamiento, 10 000 de validación y las 10 000 oficiales de prueba. `results/splits.npz` guarda los índices; al reutilizarlos se verifica que sean exactamente los generados por el protocolo para evitar mezclar ejecuciones incompatibles. La normalización es `float32 / 255.0`, en el mismo orden que el ejecutor por lotes.

In [ ]:
expected_train, expected_validation = train_test_split(
    np.arange(len(train_labels_full)), test_size=0.20, stratify=train_labels_full, random_state=SEED
)
split_path = RESULTS / "splits.npz"
if split_path.is_file():
    with np.load(split_path) as saved:
        splits = {name: saved[name] for name in ("train", "validation", "test")}
    np.testing.assert_array_equal(splits["train"], expected_train)
    np.testing.assert_array_equal(splits["validation"], expected_validation)
    np.testing.assert_array_equal(splits["test"], np.arange(len(test_raw)))
    print(f"Loaded matching split indices from {split_path}")
else:
    splits = {"train": expected_train, "validation": expected_validation, "test": np.arange(len(test_raw))}
    np.savez(split_path, **splits)
    print(f"Saved split indices to {split_path}")

def normalize(images):
    return images.astype(np.float32) / 255.0

x_train = normalize(train_raw[splits["train"]])
y_train = train_labels_full[splits["train"]]
x_val = normalize(train_raw[splits["validation"]])
y_val = train_labels_full[splits["validation"]]
x_test = normalize(test_raw)
y_test = test_labels
assert (len(y_train), len(y_val), len(y_test)) == (40000, 10000, 10000)
print("Normalized partitions:", [(array.shape, array.dtype) for array in (x_train, x_val, x_test)])

## 4. Diagnóstico: resolución y balance

Se registran forma, tipo y rango de píxeles por partición; se comprueba el conteo por clase antes de entrenar. La barra permite detectar desequilibrios que una tabla sola podría ocultar. Cada tabla mostrada se exporta a CSV y cada figura a PNG de 200 ppp.

In [ ]:
def save_table(frame, filename):
    frame.to_csv(RESULTS / filename, index=False)
    display(frame)
    return frame

def save_figure(fig, filename):
    fig.savefig(FIGURES / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(FIGURES / filename)))

resolution = pd.DataFrame([
    {"partition": "official_train", "count": len(train_raw), "shape": str(train_raw.shape[1:]), "dtype": str(train_raw.dtype), "min": int(train_raw.min()), "max": int(train_raw.max())},
    {"partition": "train", "count": len(y_train), "shape": str(x_train.shape[1:]), "dtype": str(x_train.dtype), "min": float(x_train.min()), "max": float(x_train.max())},
    {"partition": "validation", "count": len(y_val), "shape": str(x_val.shape[1:]), "dtype": str(x_val.dtype), "min": float(x_val.min()), "max": float(x_val.max())},
    {"partition": "test", "count": len(y_test), "shape": str(x_test.shape[1:]), "dtype": str(x_test.dtype), "min": float(x_test.min()), "max": float(x_test.max())},
])
save_table(resolution, "image_resolution_ranges.csv")
balance = pd.DataFrame([
    {"class_id": class_id, "class": name, "train": int(np.sum(y_train == class_id)), "validation": int(np.sum(y_val == class_id)), "test": int(np.sum(y_test == class_id))}
    for class_id, name in enumerate(CLASS_NAMES)
])
save_table(balance, "dataset_summary.csv")
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=balance.melt(id_vars=["class_id", "class"], value_vars=["train", "validation", "test"], var_name="split", value_name="count"), x="class", y="count", hue="split", ax=ax)
ax.set(title="Balance por clase y partición", xlabel="Clase", ylabel="Imágenes")
ax.tick_params(axis="x", rotation=35)
save_figure(fig, "class_balance.png")

## 5. Diagnóstico: ruido/nitidez y variabilidad

La varianza del Laplaciano de cada imagen en escala de grises funciona como **indicador aproximado de nitidez**; no mide ruido de sensor ni calidad semántica por sí sola. Por clase se resumen esa medida, el brillo y contraste por imagen, y la desviación típica entre imágenes para cada píxel/canal (promediada). La cuadrícula de muestras permite contrastar las estadísticas con la diversidad visual real.

In [ ]:
train_raw_split = train_raw[splits["train"]]
sharpness = np.array([
    float(cv2.Laplacian(cv2.cvtColor(image, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var())
    for image in train_raw_split
], dtype=np.float64)
brightness = np.empty(len(train_raw_split), dtype=np.float32)
contrast = np.empty(len(train_raw_split), dtype=np.float32)
for start in range(0, len(train_raw_split), 512):
    pixels = train_raw_split[start:start + 512].astype(np.float32).reshape(-1, 32 * 32 * 3) / 255.0
    brightness[start:start + len(pixels)] = pixels.mean(axis=1)
    contrast[start:start + len(pixels)] = pixels.std(axis=1)
image_stats = pd.DataFrame({"class_id": y_train, "class": CLASS_NAMES[y_train], "laplacian_variance": sharpness, "mean_brightness": brightness, "contrast_std": contrast})
summary_rows = []
for class_id, name in enumerate(CLASS_NAMES):
    selected = image_stats.loc[image_stats.class_id == class_id]
    class_pixels = x_train[y_train == class_id]
    summary_rows.append({
        "class_id": class_id, "class": name,
        "sharpness_mean": selected.laplacian_variance.mean(), "sharpness_median": selected.laplacian_variance.median(),
        "sharpness_std": selected.laplacian_variance.std(),
        "brightness_mean": selected.mean_brightness.mean(), "brightness_std": selected.mean_brightness.std(),
        "contrast_mean": selected.contrast_std.mean(), "contrast_std": selected.contrast_std.std(),
        "intra_class_pixel_std_mean": float(class_pixels.std(axis=0).mean()),
    })
variability = save_table(pd.DataFrame(summary_rows), "class_sharpness_variability.csv")
fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=image_stats, x="class", y="laplacian_variance", color="#78a9bd", showfliers=False, ax=ax)
ax.set(title="Nitidez aproximada por clase (varianza del Laplaciano)", xlabel="Clase", ylabel="Varianza del Laplaciano")
ax.tick_params(axis="x", rotation=35)
save_figure(fig, "sharpness_boxplot.png")
fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
for ax, column, title in zip(axes, ("mean_brightness", "contrast_std"), ("Brillo medio por clase", "Contraste por clase")):
    sns.boxplot(data=image_stats, x="class", y=column, color="#78a9bd", showfliers=False, ax=ax)
    ax.set(title=title, xlabel="Clase", ylabel="Valor normalizado")
    ax.tick_params(axis="x", rotation=35)
save_figure(fig, "brightness_contrast.png")
fig, axes = plt.subplots(len(CLASS_NAMES), 5, figsize=(10, 16.5))
for class_id, name in enumerate(CLASS_NAMES):
    indices = np.flatnonzero(y_train == class_id)[:5]
    for column, ax in enumerate(axes[class_id]):
        ax.imshow(train_raw_split[indices[column]])
        ax.axis("off")
        if column == 0:
            ax.set_ylabel(name, rotation=0, labelpad=38, va="center")
fig.suptitle("Muestras de entrenamiento por clase")
save_figure(fig, "class_samples.png")

## 6. Diagnóstico: candidatos a duplicados entre entrenamiento oficial y prueba

Se aplica pHash a **todas** las 50 000 imágenes oficiales de entrenamiento y las 10 000 de prueba. Un índice de cinco fragmentos de 64 bits garantiza que todo par con distancia Hamming ≤ 4 comparta al menos un fragmento; los candidatos se verifican con el hash completo. Estos pares **no son duplicados confirmados**. El cribado previo encontró 117 pares; aquí se recalcula y se muestran ejemplos sin asumir ese resultado.

In [ ]:
def phash_candidates(train_images, test_images, max_distance=4):
    if not 0 <= max_distance <= 4:
        raise ValueError("Five chunks support a threshold up to four bits")
    train_hashes = [int(str(imagehash.phash(PILImage.fromarray(image))), 16) for image in train_images]
    test_hashes = [int(str(imagehash.phash(PILImage.fromarray(image))), 16) for image in test_images]
    widths, shifts = (13, 13, 13, 13, 12), (0, 13, 26, 39, 52)
    index = defaultdict(list)
    for train_id, value in enumerate(train_hashes):
        for chunk_id, (width, shift) in enumerate(zip(widths, shifts)):
            index[(chunk_id, (value >> shift) & ((1 << width) - 1))].append(train_id)
    pairs = []
    for test_id, value in enumerate(test_hashes):
        candidates = set()
        for chunk_id, (width, shift) in enumerate(zip(widths, shifts)):
            candidates.update(index[(chunk_id, (value >> shift) & ((1 << width) - 1))])
        for train_id in candidates:
            distance = (train_hashes[train_id] ^ value).bit_count()
            if distance <= max_distance:
                pairs.append({"train_index": train_id, "test_index": test_id, "hamming_distance": distance,
                              "train_class": CLASS_NAMES[train_labels_full[train_id]], "test_class": CLASS_NAMES[test_labels[test_id]]})
    return pd.DataFrame(pairs)

duplicate_pairs = save_table(phash_candidates(train_raw, test_raw), "phash_train_test_candidate_pairs.csv")
duplicate_summary = save_table(pd.DataFrame([{"official_train_count": len(train_raw), "test_count": len(test_raw), "max_hamming_distance": 4, "candidate_pair_count": len(duplicate_pairs), "status": "unconfirmed candidates"}]), "phash_screen_summary.csv")
print(f"pHash candidate pairs: {len(duplicate_pairs)}")
examples = duplicate_pairs.head(8)
fig, axes = plt.subplots(max(1, len(examples)), 2, figsize=(5, 2.2 * max(1, len(examples))), squeeze=False)
for row, pair in enumerate(examples.itertuples()):
    axes[row, 0].imshow(train_raw[pair.train_index])
    axes[row, 1].imshow(test_raw[pair.test_index])
    axes[row, 0].set_title(f"Train #{pair.train_index} (d={pair.hamming_distance})", fontsize=9)
    axes[row, 1].set_title(f"Test #{pair.test_index}", fontsize=9)
if examples.empty:
    for ax in axes[0]:
        ax.text(0.5, 0.5, "Sin candidatos", ha="center", va="center")
for ax in axes.flat:
    ax.axis("off")
fig.suptitle("Ejemplos de pares candidatos a duplicados próximos")
save_figure(fig, "near_duplicate_candidates.png")

### Análisis (escribir por el estudiante)

- ¿Qué indican la resolución, el rango de píxeles y la distribución de clases para la comparación?
- ¿Cómo varían la nitidez aproximada, el brillo, el contraste y la diversidad visual entre clases?
- ¿Qué evidencia adicional se necesitaría para confirmar o descartar los pares candidatos de pHash?

## 7. Transformaciones y semántica de clase

| Condición | Transformación | Justificación o riesgo semántico |
|---|---|---|
| `reference` | Ninguna | Control: arquitectura original sin bloque estocástico. |
| `full` | Reflejo horizontal + rotación ±36° + zoom 10 % | Recupera el bloque del Momento 3. |
| `flip_h` | Reflejo horizontal | Suele conservar la clase de animales y vehículos. |
| `rotation` | `RandomRotation(0.1)` = ±36° | Evalúa tolerancia de orientación, aunque puede generar poses poco frecuentes. |
| `zoom` | `RandomZoom(0.1)` ≈ 10 % | Evalúa variación de escala sin modificar la etiqueta prevista. |
| `flip_v` | Reflejo vertical | Sonda cuestionable: animales y vehículos invertidos quedan fuera de distribución. |
| `blur` | Desenfoque gaussiano, σ ∈ [0.5, 1.5] | Sonda que puede destruir detalle fino a 32 × 32. |

La cuadrícula usa las mismas seis imágenes de entrenamiento en todas las condiciones. Las transformaciones se aplican **dentro del modelo** y solo durante `training=True`.

In [ ]:
transform_table = pd.DataFrame([
    ("reference", "none", "Control without stochastic transformation"),
    ("full", "horizontal flip + rotation ±36° + zoom 10%", "Momento 3 embedded block"),
    ("flip_h", "horizontal flip", "Usually preserves animal and vehicle class"),
    ("rotation", "RandomRotation(0.1): ±36°", "Tests orientation tolerance"),
    ("zoom", "RandomZoom(0.1): about 10%", "Tests scale variation"),
    ("flip_v", "vertical flip", "Questionable: upside-down animals and vehicles are out of distribution"),
    ("blur", "Gaussian blur, sigma 0.5–1.5", "May erase class-discriminating detail at 32×32"),
], columns=["scenario", "transformation", "semantic_rationale"])
save_table(transform_table, "transformation_rationale.csv")

class RandomGaussianBlur(layers.Layer):
    """Training-only depthwise Gaussian blur with a sampled sigma."""
    def __init__(self, seed=42, sigma_min=0.5, sigma_max=1.5, **kwargs):
        super().__init__(**kwargs)
        self.seed, self.sigma_min, self.sigma_max = seed, sigma_min, sigma_max
        self.generator = tf.keras.random.SeedGenerator(seed)

    def call(self, inputs, training=None):
        def apply_blur():
            sigma = tf.keras.random.uniform((), self.sigma_min, self.sigma_max, seed=self.generator)
            coords = tf.range(-3, 4, dtype=tf.float32)
            kernel_1d = tf.exp(-0.5 * tf.square(coords / sigma))
            kernel_1d /= tf.reduce_sum(kernel_1d)
            kernel_2d = tf.tensordot(kernel_1d, kernel_1d, axes=0)
            kernel = tf.reshape(kernel_2d, (7, 7, 1, 1))
            kernel = tf.tile(kernel, (1, 1, tf.shape(inputs)[-1], 1))
            padded = tf.pad(inputs, ((0, 0), (3, 3), (3, 3), (0, 0)), mode="REFLECT")
            return tf.nn.depthwise_conv2d(padded, kernel, strides=[1, 1, 1, 1], padding="VALID")
        if isinstance(training, bool):
            return apply_blur() if training else inputs
        return tf.cond(tf.cast(training if training is not None else False, tf.bool), apply_blur, lambda: inputs)

    def get_config(self):
        return {**super().get_config(), "seed": self.seed, "sigma_min": self.sigma_min, "sigma_max": self.sigma_max}

def build_augmentation(scenario, seed=42):
    if scenario not in ALL_SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")
    builders = {
        "flip_h": lambda: layers.RandomFlip("horizontal", seed=seed),
        "rotation": lambda: layers.RandomRotation(0.1, seed=seed),
        "zoom": lambda: layers.RandomZoom(0.1, seed=seed),
        "flip_v": lambda: layers.RandomFlip("vertical", seed=seed),
        "blur": lambda: RandomGaussianBlur(seed=seed),
    }
    components = ("flip_h", "rotation", "zoom") if scenario == "full" else (() if scenario == "reference" else (scenario,))
    transforms = [builders[name]() for name in components]
    if not transforms:
        transforms = [layers.Identity()]
    return tf.keras.Sequential(transforms, name=f"augmentation_{scenario}")

## 8. Protocolo equivalente y CNN

Las siete condiciones comparten exactamente la CNN del Momento 3, Adam a 10⁻³, entropía cruzada categórica dispersa, lote 128, barajado de 10 000, máximo 30 épocas, `EarlyStopping(val_loss, patience=6, restore_best_weights=True)` y `ReduceLROnPlateau(val_loss, factor=0.5, patience=2, min_lr=1e-6)`. Solo cambia el bloque previo de aumento. Se imprime `model.summary()` y se verifica que cada condición tenga **290 090 parámetros**.

In [ ]:
protocol = pd.DataFrame([
    ("Seed", SEED), ("Train/validation/test", f"{len(y_train)}/{len(y_val)}/{len(y_test)}"),
    ("Normalization", "float32 / 255.0"), ("Epoch limit", EPOCHS), ("Batch size", BATCH_SIZE),
    ("Optimizer", "Adam, learning rate 1e-3"), ("Loss", "sparse_categorical_crossentropy"),
    ("EarlyStopping", "val_loss, patience=6, restore_best_weights=True"),
    ("ReduceLROnPlateau", "val_loss, factor=0.5, patience=2, min_lr=1e-6"),
    ("tf.data shuffle", "buffer_size=10000, seed per run"), ("Mixed precision", "disabled"),
], columns=["parameter", "value"])
save_table(protocol, "experiment_protocol.csv")

def build_cnn_model(augmentation=None, input_shape=(32, 32, 3), num_classes=10):
    if augmentation is None:
        augmentation = build_augmentation("reference")
    inputs = layers.Input(shape=input_shape)
    x = augmentation(inputs)
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return models.Model(inputs=inputs, outputs=outputs, name="CNN_CIFAR10_Reproducible")

model = build_cnn_model(build_augmentation("full", seed=SEED))
model.summary()
parameter_table = []
for scenario in ALL_SCENARIOS:
    candidate = build_cnn_model(build_augmentation(scenario, seed=SEED))
    count = candidate.count_params()
    assert count == 290090, (scenario, count)
    parameter_table.append({"scenario": scenario, "parameters": count})
save_table(pd.DataFrame(parameter_table), "model_parameter_counts.csv")
del model, candidate
tf.keras.backend.clear_session()
gc.collect()

## 9. Comprobación de aumento exclusivo en entrenamiento y ejemplos visuales

Para cada bloque se comprueba igualdad exacta entre la entrada y la salida con `training=False`, además de dos inferencias deterministas de la CNN. Los conjuntos de validación y prueba se construyen sin aumento; el test nunca se transforma. En la cuadrícula, las transformaciones se activan explícitamente con `training=True` solo sobre ejemplos del entrenamiento.

In [ ]:
example_indices = [int(np.flatnonzero(y_train == class_id)[0]) for class_id in range(6)]
example_images = x_train[example_indices]
for scenario in ALL_SCENARIOS:
    augmentation = build_augmentation(scenario, seed=SEED)
    candidate = build_cnn_model(augmentation)
    sample = tf.convert_to_tensor(example_images, dtype=tf.float32)
    np.testing.assert_array_equal(augmentation(sample, training=False).numpy(), sample.numpy())
    np.testing.assert_array_equal(candidate(sample, training=False).numpy(), candidate(sample, training=False).numpy())
    del candidate, augmentation
print("Train-only and deterministic-inference assertions passed for all seven scenarios; test remains unaugmented.")
tf.keras.backend.clear_session()
fig, axes = plt.subplots(len(ALL_SCENARIOS), len(example_indices), figsize=(12, 1.8 * len(ALL_SCENARIOS)), squeeze=False)
for row, scenario in enumerate(ALL_SCENARIOS):
    augmented = build_augmentation(scenario, seed=SEED)(example_images, training=True).numpy()
    for column, ax in enumerate(axes[row]):
        ax.imshow(np.clip(augmented[column], 0, 1))
        ax.axis("off")
        if row == 0:
            ax.set_title(CLASS_NAMES[y_train[example_indices[column]]], fontsize=9)
        if column == 0:
            ax.set_ylabel(scenario, rotation=0, labelpad=55, va="center")
fig.suptitle("Las mismas seis imágenes de entrenamiento bajo cada condición")
save_figure(fig, "augmentation_grid.png")

## 10. Entrenamiento reanudable y cálculo de métricas

El manifiesto incorpora escenario, semilla, hiperparámetros, conteos y SHA-256 de las matrices **normalizadas** de entrenamiento, validación y prueba, con sus etiquetas, en ese orden, incluyendo forma y `dtype`. Solo se reutilizan `history.json`, `metrics.json` y `predictions.npz` si están íntegros y el manifiesto coincide exactamente. Una discrepancia detiene la ejecución sin modificar la carpeta afectada. Una ejecución faltante se entrena, salvo que `NB_REUSE_ONLY=1` exija reutilización. El marcador de progreso y el manifiesto final mantienen el mismo formato que el ejecutor por lotes.

In [ ]:
class EpochTimer(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.seconds = []
    def on_epoch_begin(self, epoch, logs=None):
        self.started = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        self.seconds.append(time.perf_counter() - self.started)

def make_dataset(images, labels, batch_size, seed=None):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    if seed is not None:
        dataset = dataset.shuffle(10000, seed=seed)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def write_json(path, content):
    path.write_text(json.dumps(content, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def dataset_fingerprint(partitions):
    digest = hashlib.sha256()
    for array in partitions:
        value = np.asarray(array)
        digest.update(f"{value.shape}:{value.dtype};".encode("ascii"))
        digest.update(memoryview(np.ascontiguousarray(value)).cast("B"))
    return digest.hexdigest()

def artifacts_valid(history_path, metrics_path, predictions_path):
    if not all(path.is_file() for path in (history_path, metrics_path, predictions_path)):
        return False
    try:
        json.loads(history_path.read_text(encoding="utf-8"))
        json.loads(metrics_path.read_text(encoding="utf-8"))
        with np.load(predictions_path) as predictions:
            return {"prediction", "probability", "truth"}.issubset(predictions.files)
    except (OSError, ValueError, EOFError, zipfile.BadZipFile):
        return False

def balanced_subset_indices(labels, per_class, seed=42):
    if per_class <= 0:
        raise ValueError("per_class must be positive")
    rng = np.random.default_rng(seed)
    pieces = []
    for class_id in range(len(CLASS_NAMES)):
        available = np.flatnonzero(labels == class_id)
        if len(available) < per_class:
            raise ValueError(f"Class {class_id} has only {len(available)} training rows")
        pieces.append(rng.choice(available, size=per_class, replace=False))
    return np.sort(np.concatenate(pieces))

### Ejecución de una condición

Se copian la canalización `tf.data`, la CNN, las dos devoluciones de llamada y las métricas del protocolo por lotes. La exactitud, F1 macro, métricas por clase y matriz de confusión se calculan sobre la prueba oficial intacta. Los pesos restaurados corresponden a la mejor pérdida de validación.

In [ ]:
def train_scenario(scenario, seed, train_images, train_labels, val_images, val_labels, test_images, test_labels,
                   output_root=RESULTS, epochs=30, batch_size=128, subset_per_class=None, fingerprint=None):
    if scenario not in ALL_SCENARIOS:
        raise ValueError(f"Unknown scenario: {scenario}")
    run_dir = output_root / "runs" / f"{scenario}_seed{seed}"
    history_path, metrics_path, predictions_path = (run_dir / name for name in ("history.json", "metrics.json", "predictions.npz"))
    manifest_path, marker_path = run_dir / "run_config.json", run_dir / ".in_progress.json"
    if fingerprint is None:
        fingerprint = dataset_fingerprint((train_images, train_labels, val_images, val_labels, test_images, test_labels))
    config = {
        "version": 1, "scenario": scenario, "seed": int(seed), "epochs": int(epochs), "batch_size": int(batch_size),
        "subset_per_class": subset_per_class, "train_count": len(train_labels), "validation_count": len(val_labels),
        "test_count": len(test_labels), "dataset_sha256": fingerprint,
    }
    valid = artifacts_valid(history_path, metrics_path, predictions_path)
    if manifest_path.is_file() or marker_path.is_file():
        prior_path = manifest_path if manifest_path.is_file() else marker_path
        try:
            prior = json.loads(prior_path.read_text(encoding="utf-8"))
        except (OSError, ValueError, json.JSONDecodeError) as error:
            raise ValueError(f"Unreadable run configuration at {prior_path}; inspect this run before reuse") from error
        if prior != config:
            changed = ", ".join(key for key in config if prior.get(key) != config[key])
            raise ValueError(f"Existing run configuration differs at {run_dir}: {changed}. Use a separate output directory or remove the stale run explicitly.")
        if manifest_path.is_file() and valid and not FORCE_RETRAIN:
            print(f"Skipping completed run: {run_dir} (configuration matched)")
            return json.loads(metrics_path.read_text(encoding="utf-8"))
    elif valid:
        raise ValueError(f"Complete legacy run at {run_dir} has no configuration manifest; inspect or move it before reuse.")
    if REUSE_ONLY:
        raise RuntimeError(f"Reuse-only verification forbids training: {run_dir}")
    run_dir.mkdir(parents=True, exist_ok=True)
    manifest_path.unlink(missing_ok=True)
    write_json(marker_path, config)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.random.set_seed(seed)
    tf.keras.backend.clear_session()
    augmentation = build_augmentation(scenario, seed=seed)
    model = build_cnn_model(augmentation)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    timer = EpochTimer()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
        timer,
    ]
    train_ds = make_dataset(train_images, train_labels, batch_size, seed=seed)
    val_ds = make_dataset(val_images, val_labels, batch_size)
    test_ds = make_dataset(test_images, test_labels, batch_size)
    started = time.perf_counter()
    history = model.fit(train_ds, epochs=epochs, validation_data=val_ds, callbacks=callbacks, verbose=1).history
    fit_seconds = time.perf_counter() - started
    probability = model.predict(test_ds, verbose=0).astype(np.float32)
    prediction = probability.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(test_labels, prediction, labels=np.arange(len(CLASS_NAMES)), zero_division=0)
    per_class = [
        {"class_id": int(i), "class": str(CLASS_NAMES[i]), "precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i]), "support": int(support[i])}
        for i in range(len(CLASS_NAMES))
    ]
    best_epoch_index = int(np.argmin(history["val_loss"]))
    metrics = {
        "scenario": scenario, "seed": int(seed), "accuracy": float(accuracy_score(test_labels, prediction)),
        "macro_f1": float(f1_score(test_labels, prediction, average="macro", zero_division=0)),
        "per_class": per_class,
        "confusion_matrix": confusion_matrix(test_labels, prediction, labels=np.arange(len(CLASS_NAMES))).tolist(),
        "best_epoch": best_epoch_index + 1,
        "best_epoch_train_accuracy": float(history["accuracy"][best_epoch_index]),
        "best_epoch_val_accuracy": float(history["val_accuracy"][best_epoch_index]),
        "generalization_gap": float(history["accuracy"][best_epoch_index] - history["val_accuracy"][best_epoch_index]),
        "final_train_loss": float(history["loss"][-1]), "final_val_loss": float(history["val_loss"][-1]),
        "epochs_trained": len(history["loss"]), "epoch_seconds": [float(value) for value in timer.seconds],
        "fit_seconds": float(fit_seconds),
    }
    write_json(history_path, {"history": {key: [float(value) for value in values] for key, values in history.items()}, "epoch_seconds": metrics["epoch_seconds"]})
    np.savez_compressed(predictions_path, prediction=prediction, probability=probability, truth=test_labels)
    write_json(metrics_path, metrics)
    pending_manifest = run_dir / "run_config.json.tmp"
    write_json(pending_manifest, config)
    pending_manifest.replace(manifest_path)
    marker_path.unlink(missing_ok=True)
    print(f"{scenario} seed={seed}: accuracy={metrics['accuracy']:.4f}, epochs={metrics['epochs_trained']}, epoch_seconds={metrics['epoch_seconds']}")
    del model
    gc.collect()
    return metrics

### Bucle de siete escenarios y resumen de ejecución

El valor predeterminado ejecuta los siete escenarios con semilla 42. Antes de cada uno se comprueba el manifiesto; el resumen indica épocas, mejor época y tiempo de ajuste. La ejecución de verificación usa únicamente las cuatro condiciones ya terminadas y falla antes de entrenar si alguna no coincide. Las tablas agregadas conservan los nombres del ejecutor por lotes.

In [ ]:
train_images_for_runs, train_labels_for_runs = x_train, y_train
if SUBSET_PER_CLASS is not None:
    selected = balanced_subset_indices(y_train, SUBSET_PER_CLASS, seed=SEED)
    train_images_for_runs, train_labels_for_runs = x_train[selected], y_train[selected]
fingerprint = dataset_fingerprint((train_images_for_runs, train_labels_for_runs, x_val, y_val, x_test, y_test))
print("Dataset SHA-256:", fingerprint)
records = [train_scenario(
    scenario, SEED, train_images_for_runs, train_labels_for_runs, x_val, y_val, x_test, y_test,
    output_root=RESULTS, epochs=EPOCHS, batch_size=BATCH_SIZE,
    subset_per_class=SUBSET_PER_CLASS, fingerprint=fingerprint,
) for scenario in SCENARIOS]
run_metrics = save_table(pd.DataFrame([
    {key: record[key] for key in ("scenario", "seed", "accuracy", "macro_f1", "epochs_trained", "fit_seconds")}
    for record in records
]), "run_metrics.csv")
summary = run_metrics.groupby("scenario", sort=False)[["accuracy", "macro_f1"]].agg(["mean", "std"])
summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
summary = save_table(summary.fillna(0.0).reset_index(), "metrics.csv")
gaps = save_table(pd.DataFrame([
    {key: record[key] for key in ("scenario", "seed", "best_epoch", "best_epoch_train_accuracy", "best_epoch_val_accuracy", "generalization_gap", "final_train_loss", "final_val_loss", "epochs_trained")}
    for record in records
]), "generalization_gaps.csv")
per_class = save_table(pd.DataFrame([
    {"scenario": record["scenario"], "seed": record["seed"], **item}
    for record in records for item in record["per_class"]
]), "per_class_metrics.csv")
training_summary = save_table(pd.DataFrame([
    {"scenario": record["scenario"], "epochs_trained": record["epochs_trained"], "best_epoch": record["best_epoch"], "fit_seconds": record["fit_seconds"], "mean_epoch_seconds": float(np.mean(record["epoch_seconds"]))}
    for record in records
]), "training_summary.csv")

## 11. Resultados globales y curvas de aprendizaje

La tabla expresa exactitud y F1 macro sobre prueba, con diferencias frente a la referencia en **puntos porcentuales**. La cuadrícula muestra, para **todas las condiciones ejecutadas**, pérdida y exactitud de entrenamiento y validación. No se infiere aún una conclusión causal: las diferencias se describen cuantitativamente y se discuten en los espacios del estudiante.

In [ ]:
reference_record = next(record for record in records if record["scenario"] == "reference")
main_metrics = save_table(pd.DataFrame([
    {"scenario": record["scenario"], "accuracy": record["accuracy"], "macro_f1": record["macro_f1"],
     "delta_accuracy_pp": 100 * (record["accuracy"] - reference_record["accuracy"]),
     "delta_macro_f1_pp": 100 * (record["macro_f1"] - reference_record["macro_f1"])}
    for record in records
]), "scenario_comparison.csv")
fig, axes = plt.subplots(len(SCENARIOS), 2, figsize=(12, 3 * len(SCENARIOS)), squeeze=False, constrained_layout=True)
for row, scenario in enumerate(SCENARIOS):
    path = RESULTS / "runs" / f"{scenario}_seed{SEED}" / "history.json"
    history = json.loads(path.read_text(encoding="utf-8"))["history"]
    epochs_axis = np.arange(1, len(history["loss"]) + 1)
    for column, (train_key, val_key, title) in enumerate((("loss", "val_loss", "Pérdida"), ("accuracy", "val_accuracy", "Exactitud"))):
        ax = axes[row, column]
        ax.plot(epochs_axis, history[train_key], label="Entrenamiento")
        ax.plot(epochs_axis, history[val_key], linestyle="--", label="Validación")
        ax.set(title=f"{scenario} · {title}", xlabel="Época", ylabel=title)
        ax.legend(fontsize=8)
save_figure(fig, "learning_curves_all_scenarios.png")

## 12. Brecha de generalización

La brecha se calcula en la mejor época de validación como exactitud de entrenamiento menos exactitud de validación. Se muestran además las pérdidas de la última época, la duración y si el paro temprano se activó antes del máximo de 30 épocas. Que la brecha sea menor no implica por sí solo mejor F1 en prueba.

In [ ]:
gap_report = gaps.copy()
gap_report["generalization_gap_pp"] = 100 * gap_report["generalization_gap"]
gap_report["early_stopping_fired"] = gap_report["epochs_trained"] < EPOCHS
gap_report = gap_report[["scenario", "seed", "best_epoch", "best_epoch_train_accuracy", "best_epoch_val_accuracy", "generalization_gap_pp", "final_train_loss", "final_val_loss", "epochs_trained", "early_stopping_fired"]]
save_table(gap_report, "generalization_gap_report.csv")

## 13. Matrices de confusión y métricas por clase

Se comparan matrices de confusión normalizadas por clase real (`reference` frente a `full`), precisión/recobrado/F1 por clase, variación de F1 y mapa de calor del F1 de todas las condiciones disponibles. Los pares de confusión más frecuentes excluyen la diagonal y conservan la dirección **clase real → predicción**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for ax, scenario, title in zip(axes, ("reference", "full"), ("Referencia", "Aumento completo")):
    record = next(item for item in records if item["scenario"] == scenario)
    matrix = np.asarray(record["confusion_matrix"], dtype=np.float64)
    matrix /= np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set(title=title, xlabel="Predicción", ylabel="Clase real")
    ax.tick_params(axis="x", rotation=45)
save_figure(fig, "confusion_matrices_reference_vs_full.png")
ref_class = per_class[per_class.scenario == "reference"].drop(columns=["scenario", "seed", "support"]).rename(columns={"precision": "precision_reference", "recall": "recall_reference", "f1": "f1_reference"})
full_class = per_class[per_class.scenario == "full"].drop(columns=["scenario", "seed", "support"]).rename(columns={"precision": "precision_full", "recall": "recall_full", "f1": "f1_full"})
per_class_comparison = ref_class.merge(full_class, on=["class_id", "class"])
per_class_comparison["delta_f1_pp"] = 100 * (per_class_comparison["f1_full"] - per_class_comparison["f1_reference"])
save_table(per_class_comparison, "per_class_reference_vs_full.csv")
fig, ax = plt.subplots(figsize=(11, 4))
delta = per_class_comparison["delta_f1_pp"].to_numpy()
ax.bar(CLASS_NAMES, delta, color=np.where(delta >= 0, "#478f7d", "#ad6565"))
ax.axhline(0, color="black", linewidth=0.7)
ax.set(title="Cambio F1 por clase: aumento completo − referencia", xlabel="Clase", ylabel="ΔF1 (puntos porcentuales)")
ax.tick_params(axis="x", rotation=35)
save_figure(fig, "delta_f1_per_class.png")
f1_matrix = per_class.pivot(index="class", columns="scenario", values="f1").reindex(index=CLASS_NAMES, columns=SCENARIOS)
save_table(f1_matrix.reset_index(), "per_class_f1_heatmap_values.csv")
fig, ax = plt.subplots(figsize=(max(7, len(SCENARIOS) * 1.3), 6))
sns.heatmap(f1_matrix, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax)
ax.set(title="F1 por clase y condición", xlabel="Condición", ylabel="Clase")
save_figure(fig, "per_class_f1_heatmap.png")
confusion_rows = []
for scenario in ("reference", "full"):
    matrix = np.asarray(next(item for item in records if item["scenario"] == scenario)["confusion_matrix"])
    for real in range(len(CLASS_NAMES)):
        for predicted in range(len(CLASS_NAMES)):
            if real != predicted:
                confusion_rows.append({"scenario": scenario, "true_class": CLASS_NAMES[real], "predicted_class": CLASS_NAMES[predicted], "count": int(matrix[real, predicted]), "row_fraction": float(matrix[real, predicted] / max(1, matrix[real].sum()))})
confused = pd.DataFrame(confusion_rows).sort_values(["scenario", "count"], ascending=[True, False]).groupby("scenario", sort=False).head(10).reset_index(drop=True)
save_table(confused, "top_confused_class_pairs.csv")

## 14. Ejemplos de aciertos y errores de alta confianza

Para `reference` y `full` se selecciona por **cada clase real** el acierto con mayor probabilidad máxima y el error con mayor probabilidad máxima, si existe. Se muestran etiqueta real, predicción y confianza; esto evita que las clases frecuentes dominen los ejemplos. Se leen probabilidades guardadas sobre el test sin volver a transformar imágenes.

In [ ]:
for scenario in ("reference", "full"):
    with np.load(RESULTS / "runs" / f"{scenario}_seed{SEED}" / "predictions.npz") as saved:
        truth, prediction, probability = saved["truth"], saved["prediction"], saved["probability"]
    confidence = probability.max(axis=1)
    rows = []
    fig, axes = plt.subplots(2, len(CLASS_NAMES), figsize=(2.1 * len(CLASS_NAMES), 5.4))
    for class_id in range(len(CLASS_NAMES)):
        for row, correct in enumerate((True, False)):
            candidates = np.flatnonzero((truth == class_id) & ((prediction == truth) == correct))
            ax = axes[row, class_id]
            if len(candidates):
                index = int(candidates[np.argmax(confidence[candidates])])
                ax.imshow(x_test[index])
                ax.set_title(f"{CLASS_NAMES[truth[index]]} → {CLASS_NAMES[prediction[index]]}\n{confidence[index]:.2f}", fontsize=8)
                rows.append({"scenario": scenario, "true_class": CLASS_NAMES[class_id], "outcome": "correct" if correct else "incorrect", "test_index": index, "predicted_class": CLASS_NAMES[prediction[index]], "confidence": float(confidence[index])})
            else:
                ax.text(0.5, 0.5, "Sin casos", ha="center", va="center")
            ax.axis("off")
    axes[0, 0].set_ylabel("Correctos", rotation=0, labelpad=45)
    axes[1, 0].set_ylabel("Errores", rotation=0, labelpad=45)
    fig.suptitle(f"Predicciones de alta confianza por clase real: {scenario}")
    save_figure(fig, f"prediction_examples_{scenario}.png")
    save_table(pd.DataFrame(rows), f"prediction_examples_{scenario}.csv")

### Análisis (escribir por el estudiante)

- ¿Cuánto cambian exactitud y F1 macro del aumento completo frente a la referencia, y qué muestran las curvas?
- ¿La brecha de generalización acompaña o contradice la diferencia observada en prueba?
- ¿Qué clases y pares de confusión explican el cambio? ¿Qué casos de alta confianza requieren inspección adicional?

## 15. Ablación: clasificación calculada de transformaciones

Cada condición **individual** se etiqueta por ΔF1 macro frente a `reference`, expresado en puntos porcentuales: **útil** si Δ > +0,5; **perjudicial** si Δ < −0,5; **neutra** en el intervalo cerrado [−0,5, +0,5]. Es una regla descriptiva predefinida, no una conclusión estadística ni una afirmación causal. Cuando se ejecuta un subconjunto de condiciones, solo se clasifican las presentes.

In [ ]:
ABLATION_THRESHOLD_PP = 0.5
ablation_rows = []
for scenario in ("flip_h", "rotation", "zoom", "flip_v", "blur"):
    record = next((item for item in records if item["scenario"] == scenario), None)
    if record is None:
        continue
    delta_pp = 100 * (record["macro_f1"] - reference_record["macro_f1"])
    label = "útil" if delta_pp > ABLATION_THRESHOLD_PP else ("perjudicial" if delta_pp < -ABLATION_THRESHOLD_PP else "neutra")
    ablation_rows.append({"scenario": scenario, "macro_f1": record["macro_f1"], "reference_macro_f1": reference_record["macro_f1"], "delta_macro_f1_pp": delta_pp, "classification": label, "threshold_pp": ABLATION_THRESHOLD_PP})
classification = save_table(pd.DataFrame(ablation_rows, columns=["scenario", "macro_f1", "reference_macro_f1", "delta_macro_f1_pp", "classification", "threshold_pp"]), "single_transform_classification.csv")

### Análisis (escribir por el estudiante)

- ¿Qué transformaciones individuales superan el umbral descriptivo y cuáles no?
- ¿Cómo se relaciona el resultado de `flip_v` y `blur` con sus riesgos semánticos y la resolución de 32 × 32?
- ¿Qué evidencia adicional permitiría separar variación por semilla de un efecto estable?

## Conclusiones y limitaciones (escribir por el estudiante)

- ¿Qué conclusión queda respaldada por las tablas y figuras, sin atribuir causalidad más allá del diseño comparativo?
- ¿Qué limitaciones introducen la resolución, los candidatos a duplicados, el presupuesto de entrenamiento y el uso de **una sola semilla**?
- ¿Cómo se verificaría la estabilidad con nuevas semillas o configuraciones?

**Recordatorio:** el aumento de datos no reemplaza un conjunto de datos representativo.

## Referencias

1. TensorFlow. [Data augmentation](https://www.tensorflow.org/tutorials/images/data_augmentation), tutorial oficial.
2. Dive into Deep Learning. [Image Augmentation](https://d2l.ai/chapter_computer-vision/image-augmentation.html).
3. TensorFlow. [Image classification](https://www.tensorflow.org/tutorials/images/classification), tutorial oficial.
4. Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images*. [University of Toronto](https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf).